## 1. Install RF-DETR 1.5.0

Install directly from the tagged release archive on GitHub.
Using a pinned tag ensures you get exactly the 1.5.0 feature set shown here.

## 2. Check GPU availability

RF-DETR trains on GPU when one is available and falls back to CPU otherwise.
The cell below detects your device and prints VRAM size — a useful sanity check
before choosing batch size later on.

## 3. Download COCO 2017

We use the official train/val splits — no manual splitting needed.

| Split | Images | Size |
|---|---|---|
| `train2017` | ~118 000 | ~18 GB |
| `val2017` | 5 000 | ~1 GB |
| annotations | — | ~241 MB |

### Set up the dataset directory structure

`model.train()` expects:
```
dataset/
  train/  _annotations.coco.json  + images
  valid/  _annotations.coco.json  + images
```

In [ ]:
DATASET_DIR = "coco_demo"
print("Dataset ready.")

In [ ]:
import rfdetr.datasets.aug_config as aug_config
from rfdetr.datasets.aug_config import AUG_AGGRESSIVE

for name in ("AUG_CONSERVATIVE", "AUG_AGGRESSIVE", "AUG_AERIAL", "AUG_INDUSTRIAL"):
    preset = getattr(aug_config, name)
    print(f"\n{name}:")
    for transform, params in preset.items():
        print(f"  {transform}: {params}")

In [ ]:
import os

from rfdetr import RFDETRNano  # smallest & fastest model — ideal for demos

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

model = RFDETRNano()
model.train(
    dataset_dir=str(DATASET_DIR),
    epochs=1,  # one epoch is enough to generate the grids
    batch_size=12,
    aug_config=AUG_AGGRESSIVE,
    save_dataset_grids=True,
    output_dir=OUTPUT_DIR,
    device=device,
    run_test=False,
    progress_bar=True,
)

In [ ]:
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

grids = sorted(Path(OUTPUT_DIR).glob("*_grid.jpg"))
if grids:
    fig, axes = plt.subplots(len(grids), 1, figsize=(6, 6 * len(grids)))
    if len(grids) == 1:
        axes = [axes]
    for ax, grid_path in zip(axes, grids):
        ax.imshow(mpimg.imread(grid_path))
        ax.set_title(grid_path.name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Option A: built-in preset ---
AUG_CONFIG = AUG_AGGRESSIVE

# --- Option B: extend a preset ---
# AUG_CONFIG = {**AUG_AGGRESSIVE, "VerticalFlip": {"p": 0.3}}

# --- Option C: fully custom ---
# AUG_CONFIG = {
#     "HorizontalFlip": {"p": 0.5},
#     "Rotate": {"limit": 15, "p": 0.3},
#     "RandomBrightnessContrast": {"brightness_limit": 0.2, "contrast_limit": 0.2, "p": 0.4},
#     "GaussianBlur": {"blur_limit": 3, "p": 0.2},
# }

# --- Option D: no augmentations ---
# AUG_CONFIG = {}

print("Selected augmentation config:")
for transform, params in AUG_CONFIG.items():
    print(f"  {transform}: {params}")

In [ ]:
model = RFDETRNano()
model.train(
    dataset_dir=str(DATASET_DIR),
    epochs=2,
    batch_size=24,
    aug_config=AUG_CONFIG,
    output_dir=OUTPUT_DIR,
    device=device,
    progress_bar=True,
    run_test=False,
)

In [ ]:
import requests
import supervision as sv
from PIL import Image

from rfdetr.util.coco_classes import COCO_CLASSES

image_url = "https://media.roboflow.com/dog.jpg"
image = Image.open(requests.get(image_url, stream=True).raw)

detections = model.predict(image, threshold=0.5)

labels = [COCO_CLASSES[class_id] for class_id in detections.class_id]
annotated = sv.BoxAnnotator().annotate(image.copy(), detections)
annotated = sv.LabelAnnotator().annotate(annotated, detections, labels)
sv.plot_image(annotated)